In [1]:
import warnings
warnings.filterwarnings( 'ignore' )
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import torch
import torch.nn as nn
import torch.optim as optim
import random
import time

from torch.utils.data import TensorDataset, DataLoader
from itertools import product
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, recall_score, precision_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, TimeSeriesSplit, train_test_split
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import label_binarize, LabelEncoder
from sklearn.impute import SimpleImputer

In [2]:
partition = 300

In [3]:
trainpath = f'../../../../../data/top30groups/anonLoc/scaledtrain1/train{partition}.csv'
testpath = f'../../../../../data/top30groups/anonLoc/scaledtest1/test{partition}.csv'

traindata = pd.read_csv(trainpath, encoding='ISO-8859-1')
testdata = pd.read_csv(testpath, encoding='ISO-8859-1')

In [4]:
testdata.shape

(2700, 15)

In [5]:
traindata.shape

(6300, 15)

In [6]:
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

def split_data(dftrain, dftest):
    Xtrain = dftrain.drop(columns=['gname']).values
    Ytrain = dftrain['gname'].values
    Xtest = dftest.drop(columns=['gname']).values
    Ytest = dftest['gname'].values

    # Encode labels as integers
    le = LabelEncoder()
    Ytrain = le.fit_transform(Ytrain)
    Ytest = le.transform(Ytest)

    Xtrain = Xtrain.astype(float)
    Xtest = Xtest.astype(float)

    # Convert to torch tensors and move to GPU
    Xtrain = torch.tensor(Xtrain, dtype=torch.float32).to("cuda")
    Ytrain = torch.tensor(Ytrain, dtype=torch.long).to("cuda")
    Xtest = torch.tensor(Xtest, dtype=torch.float32).to("cuda")
    Ytest = torch.tensor(Ytest, dtype=torch.long).to("cuda")

    return Xtrain, Ytrain, Xtest, Ytest, le


In [7]:
torch.cuda.empty_cache()


In [8]:
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, output_dim, activation='relu'):
        super().__init__()
        act_fn = nn.ReLU() if activation == 'relu' else nn.Tanh()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            act_fn,
            nn.Linear(hidden1, hidden2),
            act_fn,
            nn.Linear(hidden2, output_dim)
        )

    def forward(self, x):
        return self.model(x)


def train_model(model, Xtrain, Ytrain, Xval=None, Yval=None, lr=0.001, alpha=1e-4,
                searching=False, max_epochs=1000, batch_size=128, patience=50):

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=alpha)

    train_loader = DataLoader(TensorDataset(Xtrain, Ytrain), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(Xval, Yval), batch_size=batch_size) if Xval is not None else None

    best_acc = -1
    best_epoch = 0
    no_improve = 0
    best_state_dict = None
    epoch_times = []
    train_accuracies = []

    for epoch in range(max_epochs):
        start_time = time.time()
        model.train()
        total_correct, total_samples = 0, 0

        for Xbatch, Ybatch in train_loader:
            optimizer.zero_grad()
            output = model(Xbatch)
            loss = criterion(output, Ybatch)
            loss.backward()
            optimizer.step()

            #if not searching:
            preds = output.argmax(dim=1)
            total_correct += (preds == Ybatch).sum().item()
            total_samples += Ybatch.size(0)

        acc = total_correct / total_samples
        if not searching:
            train_accuracies.append(acc)

        # Validation check
        val_acc = evaluate_model(model, Xval, Yval, batch_size) if val_loader else acc

        if val_acc > best_acc:
            best_acc = val_acc
            best_epoch = epoch
            no_improve = 0
            best_state_dict = model.state_dict()
        else:
            no_improve += 1

        if epoch % 100 == 0 and epoch != 0:
            print(f"Epoch {epoch+1:03d}: loss={loss.item():.4f}, train_acc={acc:.4f}, val_acc={val_acc:.4f}, time={end_time - start_time:.2f}s")

        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1} with val_acc={best_acc:.4f}")
            break

        end_time = time.time()
        epoch_times.append(end_time - start_time)

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model, epoch_times, train_accuracies, best_epoch, best_acc




def evaluate_model(model, Xval, Yval, batch_size=128):
    model.eval()
    dataset = TensorDataset(Xval, Yval)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for Xbatch, Ybatch in loader:
            preds = model(Xbatch).argmax(dim=1)
            total_correct += (preds == Ybatch).sum().item()
            total_samples += Ybatch.size(0)

    return total_correct / total_samples

def find_best_mlp(Xtrain, Ytrain, num_classes, n_iter=20, max_epochs=1000):
    input_dim = Xtrain.shape[1]

    param_dist = {
        'hidden1': [10, 50, 100, 150, 200, 300],
        'hidden2': [25, 50, 100],
        'activation': ['relu', 'tanh'],
        'lr': [0.0001, 0.001, 0.01],
        'alpha': [1e-5, 1e-4, 1e-3, 1e-2],
        'batch_size': [128, 256, 512]
    }
    
    keys, values = zip(*param_dist.items())
    combinations = [dict(zip(keys, v)) for v in product(*values)]
    sampled_combinations = random.sample(combinations, 300)

    X_cpu = Xtrain.cpu().numpy()
    Y_cpu = Ytrain.cpu().numpy()

    # Split
    X_train_split, X_val_split, Y_train_split, Y_val_split = train_test_split(
        X_cpu, Y_cpu, test_size=0.2, random_state=42, stratify=Y_cpu
    )

    # Convert back to torch and move to device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_train_split = torch.tensor(X_train_split, dtype=torch.float32).to(device)
    Y_train_split = torch.tensor(Y_train_split, dtype=torch.long).to(device)
    X_val_split = torch.tensor(X_val_split, dtype=torch.float32).to(device)
    Y_val_split = torch.tensor(Y_val_split, dtype=torch.long).to(device)

    best_acc = -1
    best_params = None

    for i, params in enumerate(sampled_combinations):
        # Ändra sen
        print(f"Combination {i+1}/864: {params}")
        model = SimpleMLP(input_dim, params['hidden1'], params['hidden2'], num_classes, params['activation']).to(device)

        _, epoch_times, _, _, _ = train_model(model, X_train_split, Y_train_split,
                        Xval=X_val_split, Yval=Y_val_split,
                        lr=params['lr'], alpha=params['alpha'],
                        searching=True, max_epochs=max_epochs,
                        batch_size=params['batch_size'], patience=50)

        acc = evaluate_model(model, X_val_split, Y_val_split, batch_size=params['batch_size'])
        print(acc)
        if acc > best_acc:
            best_acc = acc
            best_params = params
            best_model_state = model.state_dict()


    print("Best hyperparameters:", best_params)
    print("Best acc:", best_acc)

    # Final model training on full train set
    final_model = SimpleMLP(input_dim, best_params['hidden1'], best_params['hidden2'], num_classes, best_params['activation']).to(device)
    final_model.load_state_dict(best_model_state)

    """_, epoch_times, train_accuracies, best_epoch, best_acc = train_model(
        final_model, Xtrain, Ytrain, lr=best_params['lr'], alpha=best_params['alpha'],
        searching=False, max_epochs=max_epochs
    )"""

    #print(f"Best accuracy on validation split: {best_acc * 100:.2f}%")
    print("Best hyperparameters:", best_params)
    print("Best acc:", best_acc)

    return final_model, epoch_times, best_acc


In [9]:
import torch.nn.functional as F
Xtrain, Ytrain, Xtest, Ytest, le = split_data(traindata, testdata)
best_mlp, epoch_times, best_acc = find_best_mlp(Xtrain, Ytrain, 30)

best_mlp.eval()
with torch.no_grad():
    logits = best_mlp(Xtest)
    y_pred = logits.argmax(dim=1)
    acc = (y_pred == Ytest).float().mean().item()
    pred_proba = F.softmax(logits, dim=1)
    print(f"Accuracy: {acc * 100:.2f}%")


Combination 1/864: {'hidden1': 10, 'hidden2': 100, 'activation': 'tanh', 'lr': 0.01, 'alpha': 1e-05, 'batch_size': 128}
Epoch 101: loss=1.2503, train_acc=0.6407, val_acc=0.5698, time=-0.00s
Epoch 201: loss=0.9327, train_acc=0.6875, val_acc=0.5913, time=-0.00s
Early stopping at epoch 223 with val_acc=0.5984
0.5777777777777777
Combination 2/864: {'hidden1': 10, 'hidden2': 25, 'activation': 'relu', 'lr': 0.001, 'alpha': 0.01, 'batch_size': 256}
Epoch 101: loss=2.2305, train_acc=0.3304, val_acc=0.3000, time=-0.00s
Epoch 201: loss=2.0833, train_acc=0.3617, val_acc=0.3317, time=-0.00s
Epoch 301: loss=2.2584, train_acc=0.3788, val_acc=0.3540, time=-0.00s
Epoch 401: loss=2.1911, train_acc=0.3929, val_acc=0.3643, time=-0.00s
Early stopping at epoch 472 with val_acc=0.3722
0.37222222222222223
Combination 3/864: {'hidden1': 100, 'hidden2': 100, 'activation': 'relu', 'lr': 0.0001, 'alpha': 0.0001, 'batch_size': 256}
Epoch 101: loss=2.3655, train_acc=0.3306, val_acc=0.2976, time=-0.00s
Epoch 201: l

In [10]:
from sklearn.preprocessing import label_binarize
y_true_decoded = le.inverse_transform(Ytest.cpu().numpy())
y_pred_decoded = le.inverse_transform(y_pred.cpu().numpy())
y_score = pred_proba.cpu().numpy()
y_true_bin = label_binarize(Ytest.cpu().numpy(), classes=list(range(30)))


In [11]:
import os
file_path = os.path.join("results", f"gtd{partition}.txt")

# Make sure the directory exists
os.makedirs("results", exist_ok=True)

# Write a string to the file
with open(file_path, "w") as file:
    file.write(f"Accuracy: {acc:.4f}\n")
    file.write(f"Precision weighted: {precision_score(y_true_decoded, y_pred_decoded, average='weighted'):.4f}\n")
    file.write(f"Recall weighted: {recall_score(y_true_decoded, y_pred_decoded, average='weighted'):.4f}\n")
    file.write(f"F1 Score weighted: {f1_score(y_true_decoded, y_pred_decoded, average='weighted'):.4f}\n")
    file.write(f"ROCAUC Weighted: {roc_auc_score(y_true_bin, y_score, average='weighted', multi_class='ovr'):.4f}\n")


    file.write(f"Precision micro: {precision_score(y_true_decoded, y_pred_decoded, average='micro'):.4f}\n")
    file.write(f"Recall micro: {recall_score(y_true_decoded, y_pred_decoded, average='micro'):.4f}\n")
    file.write(f"F1 Score micro: {f1_score(y_true_decoded, y_pred_decoded, average='micro'):.4f}\n")
    file.write(f"ROCAUC micro: {roc_auc_score(y_true_bin, y_score, average='micro', multi_class='ovr'):.4f}\n")

    file.write(f"Precision macro: {precision_score(y_true_decoded, y_pred_decoded, average='macro'):.4f}\n")
    file.write(f"Recall macro: {recall_score(y_true_decoded, y_pred_decoded, average='macro'):.4f}\n")
    file.write(f"F1 Score macro: {f1_score(y_true_decoded, y_pred_decoded, average='macro'):.4f}\n")
    file.write(f"ROCAUC macro: {roc_auc_score(y_true_bin, y_score, average='macro', multi_class='ovr'):.4f}\n")

with open(f"results/epoch_logs_gtd{partition}", "w") as f:
    f.write('\n'.join(str(x) for x in epoch_times))

In [12]:
print(classification_report(y_true_decoded, y_pred_decoded))

                                                  precision    recall  f1-score   support

                          Abu Sayyaf Group (ASG)       0.30      0.24      0.27        90
        African National Congress (South Africa)       0.53      0.90      0.66        90
                                Al-Qaida in Iraq       0.48      0.50      0.49        90
        Al-Qaida in the Arabian Peninsula (AQAP)       0.22      0.31      0.25        90
                                      Al-Shabaab       0.27      0.21      0.24        90
             Basque Fatherland and Freedom (ETA)       0.42      0.47      0.44        90
                                      Boko Haram       0.43      0.26      0.32        90
  Communist Party of India - Maoist (CPI-Maoist)       0.53      0.59      0.56        90
       Corsican National Liberation Front (FLNC)       0.52      0.87      0.65        90
                       Donetsk People's Republic       0.52      0.64      0.58        90
Farabundo

In [13]:
print(best_mlp)

SimpleMLP(
  (model): Sequential(
    (0): Linear(in_features=14, out_features=10, bias=True)
    (1): ReLU()
    (2): Linear(in_features=10, out_features=50, bias=True)
    (3): ReLU()
    (4): Linear(in_features=50, out_features=30, bias=True)
  )
)


In [14]:
def plot_confusion_matrix(y_true, y_pred, labels):
    from sklearn.metrics import confusion_matrix
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(18, 16))
    sns.heatmap(cm_normalized,
                annot=True,
                fmt=".2f",
                xticklabels=labels,
                yticklabels=labels,
                cmap="viridis",
                square=True,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8})

    plt.title(f"Normalized Confusion Matrix (Partition {partition})", fontsize=18)
    plt.xlabel("Predicted Label", fontsize=14)
    plt.ylabel("True Label", fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()

    # Save the figure
    save_path = f"results/confusion_matrix_partition_{partition}.png"
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Saved confusion matrix for partition {partition} to {save_path}")


In [15]:

# Get all unique class labels from the truths
class_labels = np.unique(y_true_decoded)

plot_confusion_matrix(y_true_decoded, y_pred_decoded, labels=class_labels)



Saved confusion matrix for partition 300 to results/confusion_matrix_partition_300.png
